In [1]:
import json, re, subprocess, statistics, sys, time
from datetime import datetime, timezone
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import memcap

B = repo / "build/adapters"
run_id = "p7-findings-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
raw = repo / "results/raw" / run_id
raw.mkdir(parents=True)
import hashlib
git = lambda *a: subprocess.run(["git", *a], cwd=repo, capture_output=True, text=True).stdout.strip()
meta = {"run_id": run_id, "git_sha": git("rev-parse", "HEAD"),
        "dirty": bool(git("status", "--porcelain", "src", "Makefile")),
        "adapters": {b.name: {"sha256": hashlib.sha256(b.read_bytes()).hexdigest(),
                              "built": datetime.fromtimestamp(b.stat().st_mtime, timezone.utc).isoformat()}
                     for b in sorted(B.glob("*-adapter"))}}
(raw / "meta.json").write_text(json.dumps(meta, indent=1))
print(meta)

def run(name, binary, args, cap_gib=8):
    out = raw / f"{name}.jsonl"
    t = time.monotonic()
    r = memcap.run([str(B / binary), *args, "--out", str(out)], timeout_s=3600, cap_bytes=int(cap_gib * memcap.GIB))
    (raw / f"{name}.stderr.log").write_text(r.stderr)
    recs = [json.loads(l) for l in out.read_text().splitlines()] if out.is_file() else []
    return r, recs, time.monotonic() - t

{'run_id': 'p7-findings-20261110T105629Z', 'git_sha': 'c4b68d621f291f73b70f865e069ca567de685253', 'dirty': False, 'adapters': {'apir-adapter': {'sha256': '0d79e3f0395506cafae23b51817ea587a7593d7e5d52fd801b58230f12c51e89', 'built': '2026-11-10T10:56:19.814843+00:00'}, 'simplepir-adapter': {'sha256': 'fb4ceb6f3b76a6e31b6b599768171cb1680b8b0dc0bdc277ef425160c930ff40', 'built': '2026-11-10T10:56:18.305887+00:00'}, 'vsp-adapter': {'sha256': '60839b1e22ccee61a08acd587577462dca87ef75baf1e4161e9105c909c147db', 'built': '2026-11-10T10:56:22.127963+00:00'}}}


In [2]:
# M1: upstream merkle.DecodeProof allocates as many slice headers as a corrupted proof's first 4 bytes say,
# before reading any hash. The adapter refuses those decodes (they always panic) and logs the would-be size.
args = ["--arm", "M1", "--db-bytes", "1048576", "--record-bytes", "32", "--phase", "online", "--reps", "100",
        "--warmup", "0", "--seed", "1234", "--fault", "wrong_record", "--recovery", "switch_replica", "--replicas-cheat"]
r, recs, wall = run("m1-wrong_record-cheat", "apir-adapter", args, cap_gib=2)
pat = re.compile(r"claims (\d+) hashes, more than a (\d+)-byte block holds, so merkle.DecodeProof would allocate ([0-9.]+) GiB")
claims = [(int(n), int(b), float(g)) for n, b, g in pat.findall(r.stderr)]
gib = sorted(g for _, _, g in claims)
faulted = [x for x in recs if x["faulted"]]
print(f"exit {r.returncode}, over_cap {r.over_cap}, peak footprint {r.peak_footprint_bytes / memcap.GIB:.3f} GiB, {wall:.0f} s")
print(f"{len(recs)} records, {len(faulted)} faulted, outcomes {sorted({(x['recovery_outcome'], x['recovery_attempts']) for x in faulted})}")
print(f"decodes refused: {len(claims)} (block size {sorted({b for _, b, _ in claims})} bytes)")
print(f"would-be allocation per refused decode, GiB: min {gib[0]:.1f}, median {statistics.median(gib):.1f}, max {gib[-1]:.1f}")
print(f"refused decodes that would allocate more than this Mac's 16 GiB: {sum(g > 16 for g in gib)} of {len(gib)}")
assert r.returncode == 0 and not r.over_cap and claims
assert all(x["detected"] is True and x["recovery_outcome"] == "aborted" for x in faulted)

exit 0, over_cap False, peak footprint 0.025 GiB, 5 s
100 records, 100 faulted, outcomes [('aborted', 3)]
decodes refused: 218 (block size [521] bytes)
would-be allocation per refused decode, GiB: min 0.0, median 47.2, max 94.4
refused decodes that would allocate more than this Mac's 16 GiB: 184 of 218


In [3]:
# Peak memory footprint (resident plus compressed, every 0.1 s) of each adapter at the judges' sizes.
# The judges' 2 GiB cap (VPIR_ADAPTER_MEM_CAP_GIB) rests on these.
common = ["--warmup", "0", "--seed", "1234"]
cases = {
    "M1 wrong_record switch_replica cheat": ("apir-adapter", ["--arm", "M1", "--db-bytes", "1048576", "--record-bytes", "32", "--phase", "online", "--reps", "100", "--fault", "wrong_record", "--recovery", "switch_replica", "--replicas-cheat"]),
    "M0 honest": ("apir-adapter", ["--arm", "M0", "--db-bytes", "1048576", "--record-bytes", "32", "--phase", "all", "--reps", "100", "--fault", "none"]),
    "S3 honest": ("apir-adapter", ["--arm", "S3", "--db-bytes", "1048576", "--record-bytes", "0", "--phase", "all", "--reps", "100", "--fault", "none"]),
    "S4 honest 64 KiB": ("apir-adapter", ["--arm", "S4", "--db-bytes", "65536", "--record-bytes", "0", "--phase", "all", "--reps", "20", "--fault", "none"]),
    "S0-go honest": ("simplepir-adapter", ["--arm", "S0-go", "--db-bytes", "1048576", "--record-bytes", "32", "--phase", "all", "--reps", "100", "--fault", "none"]),
    "S0-cpp honest": ("vsp-adapter", ["--arm", "S0-cpp", "--db-bytes", "1048576", "--record-bytes", "32", "--phase", "all", "--reps", "100", "--fault", "none"]),
    "S1 honest": ("vsp-adapter", ["--arm", "S1", "--db-bytes", "1048576", "--record-bytes", "32", "--phase", "all", "--reps", "100", "--fault", "none"]),
    "S2 re_preprocess 0.5": ("vsp-adapter", ["--arm", "S2", "--db-bytes", "1048576", "--record-bytes", "32", "--phase", "all", "--reps", "100", "--fault", "flip_answer_bit", "--fault-prob", "0.5", "--recovery", "re_preprocess"]),
}
rows = []
for name, (binary, args) in cases.items():
    r, recs, wall = run("peak-" + re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-"), binary, args + common)
    rows.append({"case": name, "exit": r.returncode, "over_cap": r.over_cap, "records": len(recs),
                 "peak_footprint_gib": r.peak_footprint_bytes / memcap.GIB, "wall_s": wall})
with open(raw / "peak_footprints.jsonl", "w") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")
for row in sorted(rows, key=lambda x: -x["peak_footprint_gib"]):
    print(f"{row['case']:40s} exit {row['exit']}  peak {row['peak_footprint_gib']:.2f} GiB  {row['wall_s']:.0f} s")
largest = max(rows, key=lambda x: x["peak_footprint_gib"])
print(f"largest: {largest['case']}, {largest['peak_footprint_gib']:.2f} GiB; 2 GiB cap is {2 / largest['peak_footprint_gib']:.1f}x that")
assert all(x["exit"] == 0 and not x["over_cap"] for x in rows)

S3 honest                                exit 0  peak 0.29 GiB  102 s
S2 re_preprocess 0.5                     exit 0  peak 0.24 GiB  447 s
S1 honest                                exit 0  peak 0.11 GiB  157 s
S0-go honest                             exit 0  peak 0.04 GiB  15 s
S0-cpp honest                            exit 0  peak 0.03 GiB  34 s
M1 wrong_record switch_replica cheat     exit 0  peak 0.03 GiB  4 s
S4 honest 64 KiB                         exit 0  peak 0.01 GiB  463 s
M0 honest                                exit 0  peak 0.01 GiB  0 s
largest: S3 honest, 0.29 GiB; 2 GiB cap is 6.9x that
